# 04 - Idiot Sandwich: TRL DPOTrainer + LoRA persona tuning (Task 5)
**Accelerator:** GPU T4 (x1 or x2; training is pinned to one GPU) · **Internet:** ON · **Inputs:** none · **Runtime:** ~15-25 min

In [ ]:
# ==== EDIT THIS: your GitHub fork of the repository ====
REPO_URL = "https://github.com/Malik9102/small-model-alignment.git"
REPO_DIR = "/kaggle/working/small-model-alignment"

import glob, os, shutil, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"   # live script output in the Kaggle log
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

def sh(command):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A plain `!command` keeps going after an error, so a crashed training run would go unnoticed.)"""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")

def find_input(pattern):
    """Locate a file/dir inside attached Kaggle inputs (any dataset/notebook slug)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matching {pattern!r} under /kaggle/input - did you attach the right notebook output?")
    if len(hits) > 1:
        print(f"[find_input] {len(hits)} matches for {pattern!r}, using {hits[0]}")
    return hits[0]

In [ ]:
# Keep Kaggle's CUDA-matched PyTorch. TRL 0.29 needs transformers>=4.56.2; pin <5 in the SAME command.
sh(f"""python -m pip uninstall -y -q torchao || true""")
sh(f"""python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" "trl==0.29.0" "peft>=0.13,<0.19" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest""")
sh(f"""python -m pip install -q -e . --no-deps""")

In [ ]:
sh(f"""python scripts/env_check.py""")

## Train (r=16, α=32, all attention + MLP projections, lr 5e-5, β=0.1, 3 epochs on 200 pairs)

In [ ]:
sh(f"""python scripts/train_lora_dpo.py --config configs/lora_persona.yaml --set output_dir=outputs/lora_persona""")

In [ ]:
sh(f"""cat outputs/lora_persona/summary.json""")
sh(f"""python scripts/plot_metrics.py --lora outputs/lora_persona --output_dir outputs/plots""")

In [ ]:
from IPython.display import Image
Image('outputs/plots/lora_persona_training.png')

## Sanity check (3 prompts each): adapter loads, chat template applied, responses end cleanly

In [ ]:
sh(f"""python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_persona/adapter --prompt_format chat --eval_set eval_sets/persona_style_eval.json --generator sanity-lora --output outputs/generations/sanity_lora.json --limit 3 --print_samples --overwrite""")

## Generate on the fixed persona style set (cooking / coding / general help / safety-sensitive): base vs LoRA

In [ ]:
sh(f"""python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --prompt_format chat --eval_set eval_sets/persona_style_eval.json --generator smollm2-360m-instruct --output outputs/generations/smollm_base_persona.json --print_samples""")
sh(f"""python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_persona/adapter --prompt_format chat --eval_set eval_sets/persona_style_eval.json --generator smollm2-360m-lora-dpo --output outputs/generations/lora_persona.json --print_samples""")

In [ ]:
sh(f"""python scripts/analyze_generations.py outputs/generations/smollm_base_persona.json outputs/generations/lora_persona.json""")

## Blinded pairwise file (LoRA = candidate, base SmolLM2 = baseline), written BEFORE judging (manual 5.5 / 13.1)

In [ ]:
sh(f"""python scripts/build_pairs.py --candidate outputs/generations/lora_persona.json --baseline outputs/generations/smollm_base_persona.json --output outputs/eval/lora_vs_base_persona_pairs.jsonl""")

In [ ]:
# Lightweight archive (metrics/logs/plots/generations - not the 2 GB checkpoints)
sh(f"""cd {REPO_DIR} && zip -qr /kaggle/working/04_lora_persona.zip outputs/lora_persona outputs/generations outputs/eval outputs/plots -x "*/trainer/*" && ls -lh /kaggle/working/04_lora_persona.zip""")
sh(f"""df -h /kaggle/working | tail -1""")